# Phase 4 — hindcasts for the bias correction (Colab GPU)

~780 GraphCast_small hindcasts (2016–2019, ±7 days around each event's forecast start
dates) plus the ERA5 truth they are scored against. **This takes several Colab sessions**
(~1.5 min per hindcast, ~100 per 3-h session). Everything is resumable: in every new session
run §1–§4, then §6a (and §7a if the truth cache is not finished), and check with 6b / 7b.

**Runtime:** T4 GPU. §5 runs once.

## 1. Get the code

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/RomanGorelsky/graphcast-extremes"
if os.path.exists("/content/graphcast-extremes"):
    subprocess.run(["git", "-C", "/content/graphcast-extremes", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/graphcast-extremes"], check=True)
%cd /content/graphcast-extremes
!git log -1 --format="%h %s"

## 2. Install (≈4 min), then restart

Installs the pinned requirements, swaps in the GPU build of JAX 0.7.2 and removes Colab's
newer CUDA-13 plugins. Afterwards: **Runtime → Restart session**, then continue with §3
(no reinstall needed after a restart).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!pip uninstall -y -q jax-cuda13-plugin jax-cuda13-pjrt
!pip install -q "jax[cuda12]==0.7.2"
print("Now: Runtime -> Restart session, then run section 3")

## 3. After the restart: check the GPU

In [ ]:
import os, subprocess, jax
%cd /content/graphcast-extremes
print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU - Runtime -> Change runtime type -> T4 GPU"

## 4. Storage and model files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["EXTREMES_DATA"] = "/content/data"
os.environ["EXTREMES_RESULTS"] = "/content/drive/MyDrive/graphcast-extremes/results"
!python scripts/00_download_assets.py
!python -m pytest -q tests

## 5. The plan (once)

In [ ]:
!python scripts/40_hindcast_plan.py

## 6. Hindcasts — background job (the long part)

In [ ]:
# 6a. start
LOG41 = "/content/drive/MyDrive/graphcast-extremes/results/phase4/run41.log"
!mkdir -p /content/drive/MyDrive/graphcast-extremes/results/phase4
!nohup python scripts/41_run_hindcasts.py > {LOG41} 2>&1 &
print("started; check with 6b")

In [ ]:
# 6b. progress (rerun any time)
LOG41 = "/content/drive/MyDrive/graphcast-extremes/results/phase4/run41.log"
!tail -n 3 {LOG41}
!pgrep -f "[4]1_run_hindcasts" > /dev/null && echo "STILL RUNNING" || echo "NOT RUNNING"
!free -g | head -2
!find /content/drive/MyDrive/graphcast-extremes/results/phase4/hindcasts -name "*.nc" ! -name "*.part.nc" | wc -l

## 7. ERA5 truth cache — background job, can run at the same time as §6

Reads ~1 h of ERA5 in total, one file per event.

In [ ]:
# 7a. start
LOG42 = "/content/drive/MyDrive/graphcast-extremes/results/phase4/run42.log"
!nohup python scripts/42_truth_cache.py > {LOG42} 2>&1 &
print("started; check with 7b")

In [ ]:
# 7b. progress
LOG42 = "/content/drive/MyDrive/graphcast-extremes/results/phase4/run42.log"
!tail -n 3 {LOG42}
!pgrep -f "[4]2_truth_cache" > /dev/null && echo "STILL RUNNING" || echo "NOT RUNNING"

## 8. Stop for today (optional)

Finished hindcasts are safe on Drive; only the one in progress is lost.

In [ ]:
!pkill -f "[4]1_run_hindcasts"; pkill -f "[4]2_truth_cache"; echo stopped

## 9. When everything is done: bring it to the Mac

From `MyDrive/graphcast-extremes/results/phase4/` download into
`~/Documents/GitHub/graphcast-extremes/results/phase4/`: the folders `hindcasts/` and `truth/`,
and `hindcast_plan.csv`. Then on the Mac: `python scripts/43_apply_pbc.py`.
(43 also works with a partial set: forecasts with fewer than 20 hindcasts are skipped.)